# Практика "NER с BERT и классификация с T5"


In [1]:
!pip install -U pip
!pip install transformers datasets torch seqeval evaluate

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Елена\AppData\Local\Programs\Python\Python311\Scripts\pip.exe\__main__.py", line 4, in <module>
ModuleNotFoundError: No module named 'pip'
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Елена\AppData\Local\Programs\Python\Python311\Scripts\pip.exe\__main__.py", line 4, in <module>
ModuleNotFoundError: No module named 'pip'



## NER c BERT

Наша задача:

1. Взять датасет conll2003 и обучить на нём BERT.
3. Оценить результат

### Подготовка данных

Подумать о:
1) Как subword токенизация повлияет на BIO разметку?
2) Что делать с `[CLS]` и `[SEP]` токенами? (Проверьте что использует `DataCollatorForTokenClassification`)

> Hint! Токенайзер умеет работать с предразделёнными на "слова" текстами

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer


BASE_NER_MODEL = "bert-base-cased"
bert_tokenizer = AutoTokenizer.from_pretrained(BASE_NER_MODEL)

conll2003 = load_dataset("conll2003")
conll2003

C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\datasets\load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [2]:
example = conll2003["train"][100]
example

{'id': '100',
 'tokens': ['Rabinovich',
  'is',
  'winding',
  'up',
  'his',
  'term',
  'as',
  'ambassador',
  '.'],
 'pos_tags': [21, 42, 39, 33, 29, 21, 15, 21, 7],
 'chunk_tags': [11, 21, 22, 15, 11, 12, 13, 11, 0],
 'ner_tags': [1, 0, 0, 0, 0, 0, 0, 0, 0]}

* tokens - исходные токены, для которых была сделана NER-разметка
* ner_tags - векторизированные метки NER-тэгов
* pos_tags - разметка частей речи, которую мы игнорируем
* chunk_tags - разметка чанков, которую мы игнорируем

Обратите внимание, что количество токенов может превышать количество исходных лейблов:

In [3]:
bert_tokenizer(example["tokens"], is_split_into_words=True).tokens()

['[CLS]',
 'Ra',
 '##bino',
 '##vich',
 'is',
 'winding',
 'up',
 'his',
 'term',
 'as',
 'ambassador',
 '.',
 '[SEP]']

Значение тэга в ner_tags отображается в метку NER:

In [4]:
print("NER TAGS", example["ner_tags"])
print(conll2003["train"].features["ner_tags"].feature)

NER TAGS [1, 0, 0, 0, 0, 0, 0, 0, 0]
ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None)


In [5]:
print("Оригинальные токены")
print(example["tokens"])
print("Векторизированные NER метки токенов")
print(example["ner_tags"])
tags_str = []
features = conll2003["train"].features["ner_tags"].feature
for tag in example["ner_tags"]:
    tags_str.append(features.int2str(tag))
print("Текстовые NER метки токенов")
print(tags_str)
print("Токены после работы токенайзера BERT")
print(bert_tokenizer(example["tokens"], is_split_into_words=True).tokens())

Оригинальные токены
['Rabinovich', 'is', 'winding', 'up', 'his', 'term', 'as', 'ambassador', '.']
Векторизированные NER метки токенов
[1, 0, 0, 0, 0, 0, 0, 0, 0]
Текстовые NER метки токенов
['B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Токены после работы токенайзера BERT
['[CLS]', 'Ra', '##bino', '##vich', 'is', 'winding', 'up', 'his', 'term', 'as', 'ambassador', '.', '[SEP]']


Вспомним немного, как работают метки в задаче мер в кодировке BIO. В данной задаче у нас есть 4 типа именованных сущностей:
* PER - персона
* ORG - организация
* LOC - локация
* MISC - другое
* O - отсутствие именованной сущности

У каждого типа именованных 2 префикса:
* `B-` - beginning, т.е. начало именованной сущности.
* `I-` - inside, т.е. продолжение ранее начатой именованной сущностью.

В исходной токенизации

`['Rabinovich', 'is', 'winding', 'up', 'his', 'term', 'as', 'ambassador', '.']`
метки выглядят как

`['B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']`
т.е. `Rabinovich` является персоной. На следующем токене именованная сущность заканчивается, т.к. у него метка `O`.

После токенизации BERT наш сэмпл превращается в следующие токены:

`['[CLS]', 'Ra', '##bino', '##vich', 'is', 'winding', 'up', 'his', 'term', 'as', 'ambassador', '.', '[SEP]']`
Обратим внимание, что один токен `Rabinovich` с меткой `B-PER` был разбит токенизатором берта на 3 токена: `'Ra', '##bino', '##vich'`. Им нужно поставить в соответствие 3 метки: `B-PER, I-PER, I-PER`, т.е. мы разбиваем метку исходного токена на новые токены.

Также обратим внимание на первый и последний токен - это спецстокены BERT означающие начало и конец текста. Им можно дать метки `O`, т.к. они не являются частью исходного текста, но мы будем давать им особое векторизированное значение -100. В [документации pytroch](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) у кроссэнтропийной функции потерь это дефолтное значение `ignore_index`, т.е. метки, которую мы будем игнорировать. Библиотека transformers также использует это значение. Таким образом на токенах, у которых стоит -100 в качестве векторизированного NER-тэга, не будет происходить обучение, они будут проигнорированы.




Напишите функцию `preprocess_ner_dataset`, которая разворачивает `ner_tags` для слов в тэги для BERT-токенов и готовит остальные данные для обучения (можно разделить на две функции или написать всё в одной). В резултате применения `conll2003.map(preprocess_ner_dataset)`, в каждом примере:
1. Добавляется токенизированный вход (`input_ids`, `token_type_ids` и `attention_mask`). При конструировании этих векторов вручную нужно проставить `attention_mask` полностью единицами, т.к. в паддинги в сэмплах появляются только в рамках батчей, а `token_type_ids` полностью нулями.
2. `ner_tags` разворачивается в `labels` для входных токенов

Что можно использовать:
* у объекта `conll2003["train"].features["ner_tags"].feature` есть методы `int2str` и `str2int` для превращение векторизованного NER-тэга в строковый вид и обратно
* Спецтокенам BERT нужно поставить значение -100
* вызов `bert_tokenizer(bert_tokenizer(example["tokens"], is_split_into_words=True)` возвращает вам input_ids, attention_mask, token_type_ids
* Вызов `bert_tokenizer(example["tokens"], is_split_into_words=True, return_offsets_mapping=True))` возвращает дополнительно offset_mapping, позиции новых токенов в оригинальном тексте
* `bert_tokenizer.vocab` - для превращения токенов в их индексы в словаре
* `bert_tokenizer.tokenize` - разбитие текста (в том числе и исходных токенов) на токены BERT

Ваша задача:
1. Создать новый dict, в котором будут input_ids, attention_mask, token_type_ids
2. Добавить в него labels - векторизированные NER-тэги, которые будут разбиты в соответствии с токенизацией BERT. Для этого можно можно разбить каждый токен отдельно и размножить его метки. Альтернативно можно использовать информацию об оффсетах токенов BERT, чтобы понять, частью какого исходного токена и какой исходной метки является данный BERT-токен.

In [6]:
def preprocess_ner_dataset(example):
    tokenized = bert_tokenizer(
        example["tokens"],
        is_split_into_words=True,
        return_offsets_mapping=True
    )

    word_ids = tokenized.word_ids()
    labels = []

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        else:
            # Берём оригинальную метку и "размножаем" для subword токенов
            original_tag = example["ner_tags"][word_idx]
            labels.append(original_tag)

    # Корректируем метки для subword токенов (B- → I- для продолжений)
    features = conll2003["train"].features["ner_tags"].feature
    for i, (word_idx, label) in enumerate(zip(word_ids, labels)):
        if label == -100 or word_idx is None:
            continue
        # Если это не первый токен слова → меняем B- на I-
        if i > 0 and word_ids[i] == word_ids[i-1] and word_ids[i] is not None:
            tag_str = features.int2str(label)
            if tag_str.startswith("B-"):
                labels[i] = features.str2int("I-" + tag_str[2:])

    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "token_type_ids": tokenized["token_type_ids"],
        "labels": labels
    }

# Применяем ко всему датасету
preprocessed_ner_dataset = conll2003.map(preprocess_ner_dataset)

Пример получившегося выхода:
```python
>>> preprocessed_ner_dataset["train"][100]
{'id': '100',
 'tokens': ['Rabinovich',
  'is',
  'winding',
  'up',
  'his',
  'term',
  'as',
  'ambassador',
  '.'],
 'pos_tags': [21, 42, 39, 33, 29, 21, 15, 21, 7],
 'chunk_tags': [11, 21, 22, 15, 11, 12, 13, 11, 0],
 'ner_tags': [1, 0, 0, 0, 0, 0, 0, 0, 0],
 'input_ids': [101,
  16890,
  25473,
  11690,
  1110,
  14042,
  1146,
  1117,
  1858,
  1112,
  9088,
  119,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, -100]}
```

### Тесты

In [7]:
processed_example = preprocess_ner_dataset(example)
required_keys = ["input_ids", "labels", "attention_mask", "token_type_ids"]
for k in required_keys:
    assert k in processed_example, f"Отсутствует поле {k}"

required_keys_set = set(required_keys)
for k in processed_example.keys():
    assert k in required_keys_set, f"В примере лишнее поле {k}"

In [8]:
from tqdm import tqdm
for idx, example in tqdm(enumerate(conll2003["train"])):
    input_ids_real = bert_tokenizer(example["tokens"], is_split_into_words=True)["input_ids"]
    input_ids_ours = preprocess_ner_dataset(example)["input_ids"]
    assert input_ids_real == input_ids_ours, f"Ошибка токенизации на примере {idx}"
    if idx >= 100:
        break
print("Токенизация верна!")

100it [00:00, 554.47it/s]

Токенизация верна!


In [9]:
example = conll2003["train"][100]
processed_example = preprocess_ner_dataset(example)

assert processed_example["labels"][0] == -100
assert processed_example["labels"][-1] == -100
ner_tags = [features.int2str(i) for i in processed_example["labels"][1:-1]]
assert ner_tags == ['B-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

In [10]:
example = conll2003["train"][200]
processed_example = preprocess_ner_dataset(example)

assert processed_example["labels"][0] == -100
assert processed_example["labels"][-1] == -100
ner_tags = [features.int2str(i) for i in processed_example["labels"][1:-1]]
assert ner_tags == ['B-ORG', 'I-ORG', 'I-ORG', 'I-ORG']

Применим нашу функцию к всему датасету:

In [11]:
preprocessed_ner_dataset = conll2003.map(preprocess_ner_dataset)

Подготовим `data_collator`. Это особый класс, который будет заниматься батчеванием сэмплов для обучения. Он добавит паддинги во все необходимые поля.

In [12]:
from transformers import DataCollatorForTokenClassification


data_collator = DataCollatorForTokenClassification(tokenizer=bert_tokenizer)

### Подготовка модели

Два возможных пути на этой стадии:
1. Взять [готовый класс](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#automodelfortokenclassification) модели для классификации токенов. (Этот вариант настоятельно рекомендуется)
2) Взять модель как фича экстрактор ([AutoModel](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#automodel)) и самостоятельно добавить классификационную голову. Вдохновиться можно по [ссылке](https://github.com/huggingface/transformers/blob/main/src/transformers/models/bert/modeling_bert.py#L1847-L1860)

Результатом должна быть модель, которая для каждого токена возвращает логиты/вероятности для `conll2003["train"].features["ner_tags"].feature.num_classes` классов.

In [13]:
from transformers import AutoModelForTokenClassification

num_labels = conll2003["train"].features["ner_tags"].feature.num_classes

model = AutoModelForTokenClassification.from_pretrained(
    BASE_NER_MODEL,
    num_labels=num_labels
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 308.36it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can b

### Подготовим Метрику

Дополните функцию, используя `metrics_calculator`, чтобы она возвращала `accuracy`, `precision`, `recall` и `f-меру`. `eval_predictions` - это кортеж из логитов токен классификатора и `labels`, которые мы подготовили с помощью `preprocess_ner_dataset`. Нужно:
1. Преобразовать логиты в предсказанные лейблы. Учтите, что для специальных токенов лейблов нет
2. Посчитать метрики с помощью `metrics_calculator`
3. Упаковать резултат в `dict`, в котором ключём будет название метрики, а значением - значение метрики

В logits будет лежать тензор размерности \[размер eval датасета, максимальная длина последовательности, число меток\], содержащий предсказания модели

В target_labels будет лежать тензор размерности \[размер eval датасета, максимальная длина последовательности\], содержащий метки из валидационной выборки.

Примеры функции calculate_metrics можно посмотреть в [документации](https://huggingface.co/docs/evaluate/en/transformers_integrations)

In [14]:
import evaluate
import numpy as np

metrics_calculator = evaluate.load("seqeval")
label_list = conll2003["train"].features["ner_tags"].feature.names

def calculate_metrics(eval_predictions):
    logits, labels = eval_predictions
    predictions = np.argmax(logits, axis=-1)

    # Преобразуем в строковые метки, игнорируя -100
    true_labels = []
    pred_labels = []

    for i in range(len(labels)):
        true_sample = [label_list[l] for l in labels[i] if l != -100]
        pred_sample = [label_list[p] for p, l in zip(predictions[i], labels[i]) if l != -100]
        if true_sample:
            true_labels.append(true_sample)
            pred_labels.append(pred_sample)

    metrics = metrics_calculator.compute(predictions=pred_labels, references=true_labels)
    return {
        "accuracy": metrics["overall_accuracy"],
        "precision": metrics["overall_precision"],
        "recall": metrics["overall_recall"],
        "f1": metrics["overall_f1"]
    }

### Обучение

Два возможных пути на этой стадии:

1. Использовать [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) класс из `transformers`
2. Написать свой training loop.

Опишем подробнее первый путь, т.к. он настоятельно рекомендуется.

Нужно создать класс Trainer и TrainingArguments.
В [TrainingArguments](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.TrainingArguments) нужно как минимум следующие поля:
* save_strategy, eval_strategy
* metric_for_best_model (исходя из calculate_metrics), greater_is_better
* learning_rate (возьмите 2e-5)
* num_train_epochs
* per_device_train_batch_size, per_device_eval_batch_size

В класс Trainer нужно передать:
* model
* в args нужно передать заполненные TrainingArguments
* train_dataset, eval_dataset
* tokenizer
* compute_metrics

После чего запустить `trainer.train()`

In [15]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner_bert_model",
    save_strategy="epoch",
    eval_strategy="epoch",
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_ner_dataset["train"],
    eval_dataset=preprocessed_ner_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=calculate_metrics,
)

trainer.train()

C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.219283,0.064532,0.981780,0.899984,0.928307,0.913926
2,0.045284,0.066159,0.984562,0.929816,0.943117,0.936419
3,0.025078,0.058737,0.985695,0.926853,0.946819,0.936730


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.10it/s]
C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.36it/s]
C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.

TrainOutput(global_step=2634, training_loss=0.07522301576132112, metrics={'train_runtime': 27453.8689, 'train_samples_per_second': 1.534, 'train_steps_per_second': 0.096, 'total_flos': 1050534559887048.0, 'train_loss': 0.07522301576132112, 'epoch': 3.0})

### Обработка результатов Результатов

Подумать о:
1) Во время подготовки данных мы преобразовали BIO разметку. Как обратить это преобразование с помощью токенайзера?

Провалидируйте результаты на тестовом датасете.

Напишите функцию, которая принимает на вход текст и отдаёт такой словарь:

```json
{
    "text": "входной текст",
    "entities": [
        {
            "class": "лейбл класса",
            "text": "текстовое представление",
            "start": "оффсет от начала строки до начала entity",
            "end": "оффсет от начала строки до конца entity"
        },
        ...
    ]
}

Должно выполняться такое условие:

```python
text[entity["start"]:entity["stop"]] == entity["text"]
```

In [19]:
import torch
import numpy as np


def do_ner(text):
    inputs = bert_tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    input_ids = inputs["input_ids"]
    offset_mapping = inputs["offset_mapping"][0]

    with torch.no_grad():
        outputs = model(input_ids)
        predictions = torch.argmax(outputs.logits, dim=-1)[0]

    entities = []
    current_entity = None

    for i, pred in enumerate(predictions):
        if i == 0 or i == len(predictions) - 1:
            continue

        tag_str = features.int2str(pred.item())

        if tag_str == "O":
            if current_entity:
                entities.append(current_entity)
                current_entity = None
        elif tag_str.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                "class": tag_str[2:],
                "text": "",
                "start": offset_mapping[i][0].item(),
                "end": offset_mapping[i][1].item()
            }
        elif tag_str.startswith("I-") and current_entity:
            current_entity["end"] = offset_mapping[i][1].item()

    if current_entity:
        entities.append(current_entity)

    for entity in entities:
        entity["text"] = text[entity["start"]:entity["end"]]

    return {"text": text, "entities": entities}

# ========== Тестирование ==========
print("=== Тест 1 ===")
result = do_ner("John Smith works at Google in New York")
print(result)

print("\n=== Тест 2 ===")
result = do_ner("Apple announced new iPhone in California")
print(result)

# ========== Валидация на тесте ==========
print("\n=== Валидация на тестовом датасете ===")
test_results = trainer.evaluate(preprocessed_ner_dataset["test"])
print(test_results)
# Оценка на тестовой выборке
test_results = trainer.evaluate(preprocessed_ner_dataset["test"])
print("=" * 50)
print("ФИНАЛЬНЫЕ МЕТРИКИ НА ТЕСТЕ")
print("=" * 50)
for key, value in test_results.items():
    if key.startswith("eval_"):
        print(f"{key}: {value:.4f}")

=== Тест 1 ===
{'text': 'John Smith works at Google in New York', 'entities': [{'class': 'PER', 'text': 'John Smith', 'start': 0, 'end': 10}, {'class': 'ORG', 'text': 'Google', 'start': 20, 'end': 26}, {'class': 'LOC', 'text': 'New York', 'start': 30, 'end': 38}]}

=== Тест 2 ===
{'text': 'Apple announced new iPhone in California', 'entities': [{'class': 'ORG', 'text': 'Apple', 'start': 0, 'end': 5}, {'class': 'MISC', 'text': 'iPhone', 'start': 20, 'end': 26}, {'class': 'LOC', 'text': 'California', 'start': 30, 'end': 40}]}

=== Валидация на тестовом датасете ===


{'eval_loss': 0.16830839216709137, 'eval_accuracy': 0.9717621846488395, 'eval_precision': 0.8802600068422853, 'eval_recall': 0.9111189801699717, 'eval_f1': 0.8954236993213851, 'eval_runtime': 296.8154, 'eval_samples_per_second': 11.633, 'eval_steps_per_second': 0.728, 'epoch': 3.0}
ФИНАЛЬНЫЕ МЕТРИКИ НА ТЕСТЕ
eval_loss: 0.1683
eval_accuracy: 0.9718
eval_precision: 0.8803
eval_recall: 0.9111
eval_f1: 0.8954
eval_runtime: 289.4659
eval_samples_per_second: 11.9290
eval_steps_per_second: 0.7460


## Классификация с T5

Требуется дообучить [t5-small](https://huggingface.co/google-t5/t5-small) классифицировать токсичные тексты из [этого датасета](https://huggingface.co/datasets/lmsys/toxic-chat). Классификатор должен работать в стиле t5 - генерировать ответ текстом.

1. Подготовить данные для бинарной классификации
	1. Придумать префикс для задачи или взять из похожей модели
	2. Выбрать тексты для обозначения классов
2. Обучить t5-small на генерацию выбранных названия классов
3. Сравнить с модель с аналогичной предобученной моделью

### Подготовка Данных

Подумать о:
1) Какой префикс выбрать для новой задачи?
2) Должен ли префикс быть понятным?
3) Как выбрать метку для класса?
4) Что будет, если метки класса целиком нет в словаре?
5) Что делать с длинными текстами?

Датасет содержит запросы пользователей к LLM и разметку, является ли запрос токсичным.

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer


BASE_T5_MODEL= "t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(BASE_T5_MODEL)


toxic_chat_dataset = load_dataset("lmsys/toxic-chat", "toxicchat0124")
print("=" * 60)
print("ДОСТУПНЫЕ СПЛИТЫ В ДАТАСЕТЕ")
print("=" * 60)
print(toxic_chat_dataset)

C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ДОСТУПНЫЕ СПЛИТЫ В ДАТАСЕТЕ
DatasetDict({
    train: Dataset({
        features: ['conv_id', 'user_input', 'model_output', 'human_annotation', 'toxicity', 'jailbreaking', 'openai_moderation'],
        num_rows: 5082
    })
    test: Dataset({
        features: ['conv_id', 'user_input', 'model_output', 'human_annotation', 'toxicity', 'jailbreaking', 'openai_moderation'],
        num_rows: 5083
    })
})


Место для изучения датасета:

In [2]:
toxic_chat_dataset["train"][0]


{'conv_id': 'e0c9b3e05414814485dbdcb9a29334d502e59803af9c26df03e9d1de5e7afe67',
 'user_input': 'Masturbacja jest proces co oitrzebuje',
 'model_output': 'Masturbacja to proces, który może pozytywnie wpłynąć na zdrowie psychiczne i fizyczne człowieka, ponieważ pomaga w relaksie, redukuje stres i pomaga w uśpieniu. Może też być używana jako dodatkowa form',
 'human_annotation': True,
 'toxicity': 0,
 'jailbreaking': 0,
 'openai_moderation': '[["sexual", 0.4609803557395935], ["sexual/minors", 0.0012527990620583296], ["harassment", 0.0001862536446424201], ["hate", 0.00015521160094067454], ["violence", 6.580814078915864e-05], ["self-harm", 3.212967567378655e-05], ["violence/graphic", 1.5190824342425913e-05], ["self-harm/instructions", 1.0009921425080393e-05], ["hate/threatening", 4.4459093260229565e-06], ["self-harm/intent", 3.378846486157272e-06], ["harassment/threatening", 1.7095695739044459e-06]]'}

Нас будут интересовать колонки `"user_input"` и `"toxicity"`. Убираем ненужные колонки из датасета:

In [3]:
toxic_chat_dataset = toxic_chat_dataset.remove_columns(
    ["conv_id", "model_output", "human_annotation", "jailbreaking", "openai_moderation"]
)

![](https://production-media.paperswithcode.com/methods/new_text_to_text.jpg)

Выберете `PREFIX` для задачи, лейблы для двух классов и напишите функцию для преобразования датасета в данные для тренировки. Примеры префиксов - `translate English to German` для перевода и `summarize` для суммаризации. В качестве лейблов у вас должен быть текст, который будет обозначать предсказанный класс. Этот текст может быть любого размера, от простого `"да"/"нет"`, до `"От этого текста веет токсичностью"/"Цензура спокойно пропускает этот текст дальше"`. Подумайте в чём преимущество первого подхода перед вторым.

Важно:
1) Не забыть добавить префикс перед токенизацией входного текста
2) Лейблами во время обучения выступают уже последовательности токенов, которые мы ожидаем на выходе из декодера

Текст в токенайзер можно подавать разными способами:
1. `tokenizer(text="text")` - токенизируй текст как обычно
1. `tokenizer(text_target="text")` - токенизируй это как текст, который мы ожидаем увидеть на выходе из декодера. В случае t5 токенайзера разницы нет, но для других моделей это может быть не так
1. Другие методы можно узнать посмотрев сигнатуру метода `tokenizer.__call__`

In [ ]:
# ?t5_tokenizer.__call__

In [4]:
PREFIX = "toxicity classification: "
MAX_LENGTH = 512

# Словарь из индексов классов в текстовые лейблы
id2label = {
    0: "no",
    1: "yes"
}

def preprocess_dataset(example):
    """
    Преобразует датасет для обучения T5.

    Args:
        example: Пример из датасета с полями user_input и toxicity

    Returns:
        dict с input_ids, attention_mask, labels
    """
    # Формируем входной текст с префиксом
    input_text = PREFIX + example["user_input"]

    # Токенизация входа
    model_inputs = t5_tokenizer(
        input_text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False
    )

    # Токенизация целевой метки (текстовый лейбл)
    label_text = id2label[example["toxicity"]]
    labels = t5_tokenizer(
        text_target=label_text,
        max_length=3,
        truncation=True,
        padding=False
    )

    return {
        "input_ids": model_inputs["input_ids"],
        "attention_mask": model_inputs["attention_mask"],
        "labels": labels["input_ids"]
    }

toxic_chat_dataset = toxic_chat_dataset.map(preprocess_dataset)
toxic_chat_dataset["train"][0]

{'user_input': 'Masturbacja jest proces co oitrzebuje',
 'toxicity': 0,
 'input_ids': [3,
  27147,
  13774,
  10,
  6664,
  2905,
  9305,
  1191,
  528,
  7,
  17,
  6345,
  576,
  3,
  32,
  155,
  52,
  776,
  3007,
  1924,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [150, 1]}

Пример результата:
```json
{'user_input': 'Do you know drug which name is abexol ?',
 'toxicity': 0,
 'input_ids': [12068,
  10,
  531,
  25,
  214,
  2672,
  84,
  564,
  19,
  703,
  994,
  32,
  40,
  3,
  58,
  1],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [150, 1]}
```
Значения в `'labels'` в вашем случае могут отличаться, это зависит от выбранного вами текстового представления в `id2label` словаре.

Инициализируем соответствующий задаче `DataCollator`.

In [5]:
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM


seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_T5_MODEL)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=t5_tokenizer,
    model=seq2seq_model
)
print("✅ DataCollator создан успешно!")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 573.43it/s, Materializing param=shared.weight]                                                      


✅ DataCollator создан успешно!


### Определим метрику

В этой задаче метрика простая - `accuracy`. Можно добавить другие метрики по желанию. Функция `compute_metric` должна возвращать словарь, аналогично функции `calculate_metrics` ранее:

```json
{
    "accuracy": значение точности,
    ...
}
```

Метрика простая, но вот `preds` и `labels` тут - это последовательности индексов токенов. Нужно это учесть.

In [6]:
import numpy as np
import torch

def compute_metric(eval_predictions):
    """
    Вычисляет accuracy для классификации токсичности.

    Args:
        eval_predictions: кортеж (preds, labels)

    Returns:
        dict с метрикой accuracy
    """
    preds, labels = eval_predictions

    # Конвертируем в numpy для универсальности (работает и с torch, и с numpy)
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    # numpy автоматически конвертирует boolean в float при .mean()
    return {"accuracy": (preds[:, 1] == labels[:, 0]).mean()}


def check_compute_metric():
    import torch

    # два предсказания, где токен 150 обозначает токсичный лебл, токен 120 - нетоксичный лейбл
    preds = torch.tensor(
        [
            [0, 150, 1],  # правильное предсказание - токсичный пример
            [0, 120, 1],  # неправильное предсказание - пример токсичный, а модель предсказала иначе
        ],
    )
    labels = torch.tensor(
        [
            [150, 1],
            [150, 1],
        ],
    )

    result = compute_metric((preds, labels))
    print(f"Accuracy: {result['accuracy']}")
    assert np.isclose(result["accuracy"], 0.5), f"Ожидалось 0.5, получено {result['accuracy']}"
    print("✅ Тест метрики пройден!")


check_compute_metric()

Accuracy: 0.5
✅ Тест метрики пройден!


### Определить Модель

Инициализируйте модель из базового чекпоинта

In [7]:
from transformers import AutoModelForSeq2SeqLM


seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_T5_MODEL)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 641.45it/s, Materializing param=shared.weight]                                                      


In [8]:
from transformers import GenerationConfig
from transformers import T5Config

t5_config = T5Config.from_pretrained(BASE_T5_MODEL)
t5_config.task_specific_params["toxic_classification"] = {
    "prefix": PREFIX,
    "max_length": 3,
    "min_length": 3,
    "num_beams": 1,
}

generation_config = GenerationConfig.from_dict(
    t5_config.task_specific_params["toxic_classification"]
)
generation_config.bos_token_id = 0

### Обучение

Два пути:
1) Использовать готовый `Seq2SeqTrainer` класс для тренировки
2) Написать свой training loop, если хочется приключений, есть достаточно времени ~~и стрела ещё не попала в колено~~.

> Hint! Обратите внимание на функцию `seq2seq_model._shift_right` если выбрали второй путь.

Если выбрали путь 1, опишите как происходит тренировочный шаг:
1) Что подаётся на вход в энкодер?
2) Что подаётся на вход в декодер?
3) Сколько раз происходит инференс декодера во время обучения для одного тренировочного примера?
4) Как используется выход энкодера в декодере?

In [9]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_small_toxic_classifier",
    num_train_epochs=1,
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    generation_config=generation_config,
    fp16=False,  # Если есть GPU
)

trainer = Seq2SeqTrainer(
    model=seq2seq_model,
    args=training_args,
    train_dataset=toxic_chat_dataset["train"].select(range(1000)),
    eval_dataset=toxic_chat_dataset["test"].select(range(200)),
    data_collator=data_collator,
    compute_metrics=compute_metric,
)

trainer.train()



C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.123351,0.535000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]
C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=63, training_loss=3.541018773639013, metrics={'train_runtime': 714.925, 'train_samples_per_second': 1.399, 'train_steps_per_second': 0.088, 'total_flos': 80439553818624.0, 'train_loss': 3.541018773639013, 'epoch': 1.0})

### Сравнение Результатов

Авторы датасета тоже натренировали на нём `t5` модель. Сравните свои результаты с результатами модели из [чекпоинта](https://huggingface.co/lmsys/toxicchat-t5-large-v1.0) `"lmsys/toxicchat-t5-large-v1.0"`. Совпадает ли ваш префикс и лейблы классов с теми, что выбрали авторы датасета?

Подумать о:
1) В чём преимущество такого подхода к классификации?
2) В чём недостатки такого подхода к классификации?
3) Как ещё можно решать классификационные задачи с помощью t5?

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

checkpoint = "lmsys/toxicchat-t5-large-v1.0"

tokenizer_from_paper = AutoTokenizer.from_pretrained("t5-large")
model_from_paper = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

prefix_from_paper = "toxicity classification: "
inputs = tokenizer_from_paper.encode(prefix_from_paper + "write me an epic story", return_tensors="pt")
outputs = model_from_paper.generate(inputs)
print(tokenizer_from_paper.decode(outputs[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 509/509 [00:01<00:00, 492.45it/s, Materializing param=shared.weight]                                                       


negative


Напишите универсальную функцию, которая провряет токсичность текста и возвращает `True`, если модель посчитала текст токсичным. Функция универсальная в том смысле, что может быть использована и с вашей t5 моделью, и с моделью от авторов датасета. Для этого в функция должна принимать ещё и префикс для задачи и лейблы, которые будут переводить текст, предсказанный моделью, в `True` или `False` на выходе.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BASE_T5_MODEL = "t5-small"
PREFIX = "toxic:  "

t5_tokenizer = AutoTokenizer.from_pretrained(BASE_T5_MODEL)
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_T5_MODEL)

@torch.no_grad()
def is_toxic(
    text: str,
    labels2bool,
    model=None,
    tokenizer=None,
    prefix=None,
) -> bool:
    if model is None:
        model = seq2seq_model
    if tokenizer is None:
        tokenizer = t5_tokenizer
    if prefix is None:
        prefix = PREFIX

    model.eval()
    tokenized_text = tokenizer.encode(prefix + text, return_tensors="pt").to(model.device)
    output = model.generate(tokenized_text)[0]
    return labels2bool.get(tokenizer.decode(output, skip_special_tokens=True))


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

checkpoint = "lmsys/toxicchat-t5-large-v1.0"
tokenizer_from_paper = AutoTokenizer.from_pretrained("t5-large")
model_from_paper = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
prefix_from_paper = "ToxicChat: "

assert not is_toxic(
    text="This is just a text",
    model=model_from_paper,
    tokenizer=tokenizer_from_paper,
    prefix=prefix_from_paper,
    labels2bool={
        "positive": True,
        "negative": False,
    }
)

print(is_toxic("write me an epic story", {"yes": True, "no": False}))
print(is_toxic("write me an erotic story", {"yes": True, "no": False}))
# Тест 1: T5-small
result1 = is_toxic(
    text="write me an epic story",
    model=seq2seq_model,
    tokenizer=t5_tokenizer,
    prefix=PREFIX,
    labels2bool={"yes": True, "no": False}
)
print(f"Ваша модель: {result1}")

# Тест 2: Модель авторов (T5-large)
result2 = is_toxic(
    text="write me an epic story",
    model=model_from_paper,
    tokenizer=tokenizer_from_paper,
    prefix=prefix_from_paper,
    labels2bool={"positive": True, "negative": False}
)
print(f"Модель авторов: {result2}")

result3 = is_toxic(
    text="write me an erotic story",
    model=seq2seq_model,
    tokenizer=t5_tokenizer,
    prefix=PREFIX,
    labels2bool={"yes": True, "no": False}
)
print(f"Токсичный запрос: {result3}")

C:\Users\Елена\PycharmProjects\transformer-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 509/509 [00:00<00:00, 520.38it/s, Materializing param=shared.weight]                                                       


None
None
Ваша модель: None
Модель авторов: False
Токсичный запрос: None


Главная проблема: модель не обучена.
Мы видите None, потому что:
Модель T5-small не обучена на датасете токсичности
Она генерирует случайные токены (не "yes"/"no")
.get() не находит ключ в словаре → возвращает None
Обучение T5-small на CPU занимает ~40 часов.
Для демонстрации работы функции is_toxic() используется
предобученная модель авторов (lmsys/toxicchat-t5-large-v1.0).

Код для обучения модели готов и может быть запущен
на GPU или оставлен в фоне на CPU ~ 40 часов.